# Data Cleaning

## 2.1 Introduction

### Objective

The objective of this phase is to transform the raw UAC dataset into a consistent and analysis-ready dataset. The cleaning process focuses on removing clearly unusable records, standardizing column names and data types, handling numerical formatting, checking missing values and duplicates, and validating the consistency of the observations.

In [27]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

In [28]:
df = pd.read_csv(
    "../data/raw/HHS_Unaccompanied_Alien_Children_Program.csv"
)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

display(df.head())

Dataset loaded successfully.
Shape: (1170, 6)


,Date,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care
0,"December 21, 2025",6.0,18.0,11.0,"2,484",14.0
1,"December 18, 2025",11.0,50.0,6.0,"2,472",16.0
2,"December 17, 2025",7.0,31.0,11.0,"2,481",10.0
3,"December 16, 2025",8.0,54.0,15.0,"2,468",9.0
4,"December 15, 2025",11.0,42.0,9.0,"2,470",7.0


In [29]:
original_rows = len(df)
original_columns = len(df.columns)

print("Original rows:", original_rows)
print("Original columns:", original_columns)

Original rows: 1170
Original columns: 6


## 2.2 Handling Blank Records

### Description

The raw dataset contains completely blank records that do not represent observations from the UAC program. These records are removed because they contain no analytical information.

In [30]:
blank_rows = df.isna().all(axis=1).sum()

print("Completely blank rows:", blank_rows)

df = df.dropna(how="all").copy()

print("Shape after removing blank rows:", df.shape)

Completely blank rows: 450
Shape after removing blank rows: (720, 6)


## 2.3 Standardizing Variables

### Description

The original column names are lengthy and difficult to use during analysis. They are replaced with concise and consistent variable names while preserving their original meaning.

Numerical fields are also converted into appropriate numeric data types. Comma formatting in HHS care values is removed before conversion.

In [31]:
df = df.rename(columns={
    "Date": "date",
    "Children apprehended and placed in CBP custody*": "apprehended",
    "Children in CBP custody": "cbp_custody",
    "Children transferred out of CBP custody": "transferred",
    "Children in HHS Care": "hhs_care",
    "Children discharged from HHS Care": "discharged"
})

print("Columns after renaming:")
print(df.columns.tolist())

Columns after renaming:
['date', 'apprehended', 'cbp_custody', 'transferred', 'hhs_care', 'discharged']


In [32]:
df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce"
)

print("Invalid dates:", df["date"].isna().sum())
print("Earliest date:", df["date"].min())
print("Latest date:", df["date"].max())

Invalid dates: 0
Earliest date: 2023-01-12 00:00:00
Latest date: 2025-12-21 00:00:00


In [33]:
numeric_columns = [
    "apprehended",
    "cbp_custody",
    "transferred",
    "hhs_care",
    "discharged"
]

for column in numeric_columns:
    df[column] = (
        df[column]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
    )
    
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

print("Numeric conversion completed.")

Numeric conversion completed.


In [34]:
print(df.dtypes)

date           datetime64[us]
apprehended           float64
cbp_custody           float64
transferred           float64
hhs_care                int64
discharged            float64
dtype: object


## 2.4 Missing Value Assessment

### Objective

Missing values are checked after the initial cleaning operations to ensure that the dataset does not contain incomplete observations that could affect subsequent calculations.

In [35]:
missing_summary = pd.DataFrame({
    "Missing Values": df.isna().sum(),
    "Missing Percentage": (
        df.isna().sum() / len(df) * 100
    ).round(2)
})

display(missing_summary)

,Missing Values,Missing Percentage
date,0,0.0
apprehended,0,0.0
cbp_custody,0,0.0
transferred,0,0.0
hhs_care,0,0.0
discharged,0,0.0


In [36]:
missing_rows = df[df.isna().any(axis=1)]

print("Rows containing missing values:", len(missing_rows))

display(missing_rows)

Rows containing missing values: 0


,date,apprehended,cbp_custody,transferred,hhs_care,discharged


In [37]:
df = df.sort_values("date").reset_index(drop=True)

display(df.head())
display(df.tail())

,date,apprehended,cbp_custody,transferred,hhs_care,discharged
0,2023-01-12,33.0,53.0,34.0,6566,436.0
1,2023-01-22,32.0,49.0,39.0,7122,227.0
2,2023-01-23,32.0,50.0,39.0,7280,181.0
3,2023-01-24,47.0,42.0,47.0,7433,175.0
4,2023-01-25,20.0,22.0,41.0,7538,180.0


,date,apprehended,cbp_custody,transferred,hhs_care,discharged
715,2025-12-15,11.0,42.0,9.0,2470,7.0
716,2025-12-16,8.0,54.0,15.0,2468,9.0
717,2025-12-17,7.0,31.0,11.0,2481,10.0
718,2025-12-18,11.0,50.0,6.0,2472,16.0
719,2025-12-21,6.0,18.0,11.0,2484,14.0


## 2.5 Duplicate and Consistency Checks

### Description

Duplicate dates and duplicate observations are examined to ensure that the cleaned dataset does not contain repeated records.

Additional validation checks are performed on numerical values to identify negative observations and potentially inconsistent relationships between care-stage measures.

In [38]:
duplicate_dates = df["date"].duplicated().sum()

print("Duplicate dates:", duplicate_dates)

Duplicate dates: 0


In [39]:
negative_summary = pd.DataFrame({
    "Negative Values": [
        (df[column] < 0).sum()
        for column in numeric_columns
    ]
}, index=numeric_columns)

display(negative_summary)

,Negative Values
apprehended,0
cbp_custody,0
transferred,0
hhs_care,0
discharged,0


## 2.6 Data Validation

### Description

The relationship between transfers, CBP custody, HHS care, and discharges is examined to identify observations that require further interpretation.

Because the dataset contains both population snapshots and activity measures, a value exceeding a snapshot population is not automatically treated as an error. Such observations are retained unless there is clear evidence that the record is invalid.

In [40]:
transfer_anomalies = df[
    df["transferred"] > df["cbp_custody"]
]

print(
    "Rows where transferred > CBP custody:",
    len(transfer_anomalies)
)

display(transfer_anomalies)

Rows where transferred > CBP custody: 86


,date,apprehended,cbp_custody,transferred,hhs_care,discharged
3,2023-01-24,47.0,42.0,47.0,7433,175.0
4,2023-01-25,20.0,22.0,41.0,7538,180.0
9,2023-02-02,15.0,13.0,23.0,7879,298.0
22,2023-02-22,107.0,215.0,230.0,7978,232.0
23,2023-02-23,101.0,162.0,178.0,7914,386.0
...,...,...,...,...,...,...
502,2025-01-30,47.0,42.0,47.0,3923,159.0
503,2025-02-02,20.0,22.0,41.0,3483,168.0
508,2025-02-09,15.0,13.0,23.0,2878,99.0
512,2025-02-13,15.0,10.0,23.0,2703,72.0


In [41]:
discharge_anomalies = df[
    df["discharged"] > df["hhs_care"]
]

print(
    "Rows where discharged > HHS care:",
    len(discharge_anomalies)
)

display(discharge_anomalies)

Rows where discharged > HHS care: 0


,date,apprehended,cbp_custody,transferred,hhs_care,discharged


In [42]:
df["date_gap_days"] = (
    df["date"].diff().dt.days
)

display(
    df["date_gap_days"]
    .value_counts()
    .sort_index()
)

date_gap_days
1.0     558
2.0      10
3.0     122
4.0      23
5.0       3
6.0       1
7.0       1
10.0      1
Name: count, dtype: int64

In [43]:
date_gaps = df[
    df["date_gap_days"] > 1
][
    ["date", "date_gap_days"]
]

display(date_gaps)

,date,date_gap_days
1,2023-01-22,10.0
5,2023-01-29,4.0
10,2023-02-05,3.0
15,2023-02-12,3.0
20,2023-02-20,4.0
...,...,...
703,2025-11-27,2.0
704,2025-11-30,3.0
709,2025-12-07,3.0
714,2025-12-14,3.0


In [44]:
df = df.drop(columns=["date_gap_days"])

In [45]:
print("Final dataset shape:", df.shape)

Final dataset shape: (720, 6)


In [46]:
print("Missing values:")
display(df.isna().sum())

Missing values:


date           0
apprehended    0
cbp_custody    0
transferred    0
hhs_care       0
discharged     0
dtype: int64

In [47]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate dates:", df["date"].duplicated().sum())

Duplicate rows: 0
Duplicate dates: 0


In [48]:
display(
    df[numeric_columns]
    .describe()
    .T
)

,count,mean,std,min,25%,50%,75%,max
apprehended,720.0,93.523611,72.646625,0.0,12.00,99.0,147.25,333.0
cbp_custody,720.0,171.494444,126.354965,7.0,36.00,193.0,263.25,531.0
transferred,720.0,128.668056,97.322012,0.0,14.00,157.0,199.25,440.0
hhs_care,720.0,6061.275000,2833.070109,1972.0,2467.75,6406.5,8010.25,11516.0
discharged,720.0,173.406944,125.702841,0.0,19.75,181.0,267.00,505.0


In [49]:
display(df.head(10))

,date,apprehended,cbp_custody,transferred,hhs_care,discharged
0,2023-01-12,33.0,53.0,34.0,6566,436.0
1,2023-01-22,32.0,49.0,39.0,7122,227.0
2,2023-01-23,32.0,50.0,39.0,7280,181.0
3,2023-01-24,47.0,42.0,47.0,7433,175.0
4,2023-01-25,20.0,22.0,41.0,7538,180.0
5,2023-01-29,23.0,45.0,11.0,7472,303.0
6,2023-01-30,34.0,54.0,29.0,7743,196.0
7,2023-01-31,26.0,36.0,36.0,7803,158.0
8,2023-02-01,25.0,32.0,27.0,7903,231.0
9,2023-02-02,15.0,13.0,23.0,7879,298.0


## 2.7 Final Cleaned Dataset

### Description

After completing the cleaning and validation procedures, the resulting dataset is sorted chronologically and saved as a separate processed file. The original raw dataset remains unchanged.

In [50]:
output_path = "../data/processed/uac_cleaned.csv"

df.to_csv(
    output_path,
    index=False
)

print("Cleaned dataset saved to:")
print(output_path)

Cleaned dataset saved to:
../data/processed/uac_cleaned.csv


In [51]:
import os

print(
    "File exists:",
    os.path.exists(output_path)
)

if os.path.exists(output_path):
    print(
        "File size:",
        round(os.path.getsize(output_path) / 1024, 2),
        "KB"
    )

File exists: True
File size: 27.5 KB


---

## 2.8 Key Findings

The cleaning process removes completely blank records and standardizes the dataset for analysis. Date fields and numerical variables are converted into consistent formats, column names are simplified, and duplicate and missing-value checks are completed.

The validation stage also confirms that the dataset contains both snapshot measures and activity measures, which must be interpreted differently during the analytical phases.

### Conclusion

The resulting dataset provides a consistent foundation for exploratory analysis while preserving the original observations and reporting structure.

